In [13]:

import os
from dotenv import load_dotenv

load_dotenv()

## print(os.environ["AZURE_OPENAI_API_KEY"])

True

# http request API

In [6]:

import os
import json
import requests
from dotenv import load_dotenv
load_dotenv()

headers = {
    "api-key": os.environ["AZURE_OPENAI_API_KEY"],
    "Content-Type": "application/json"
}
payload = {
    "messages": [
        {
            "role": "system",
            "content": "You are a helpful assistant."
        },
        {
            "role": "user",
            "content": "Cuál es la capital de Francia?"
        }
    ]
}
api_version = "2025-04-01-preview"
deployment_model = "gpt-4o"
chat = requests.post(
    f"https://ai-proxy.lab.epam.com/openai/deployments/{deployment_model}/chat/completions?api-version={api_version}",
    headers=headers,
    json=payload
).json()

print(json.dumps(chat, indent=3))

print(f"Response from DIAL {deployment_model}:")
print(chat["choices"][0]["message"]["content"])


{
   "id": "chatcmpl-CnTuDwhZoXtz9N1yHgMWmUCIaNm3b",
   "choices": [
      {
         "finish_reason": "stop",
         "index": 0,
         "logprobs": null,
         "message": {
            "content": "La capital de Francia es **Par\u00eds**.",
            "refusal": null,
            "role": "assistant",
            "annotations": []
         },
         "content_filter_result": {
            "error": {
               "code": "content_filter_error",
               "message": "The contents are not filtered"
            }
         },
         "content_filter_results": {}
      }
   ],
   "created": 1765909065,
   "model": "gpt-4o-2024-11-20",
   "object": "chat.completion",
   "system_fingerprint": "fp_b54fe76834",
   "usage": {
      "completion_tokens": 11,
      "prompt_tokens": 24,
      "total_tokens": 35,
      "completion_tokens_details": {
         "accepted_prediction_tokens": 0,
         "audio_tokens": 0,
         "reasoning_tokens": 0,
         "rejected_prediction_tokens

# OpenAI Python SDK

In [12]:

import os
from openai import AzureOpenAI
from dotenv import load_dotenv
load_dotenv()

client = AzureOpenAI(
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    api_version="2025-04-01-preview",
    azure_endpoint="https://ai-proxy.lab.epam.com"
)

messages = [
    {
        "role": "system",
        "content": """
        You are an Employee Training Tracker Bot. 
        You answer questions about employee training progress using the following Databricks database schema:

        Tables:
        1. employees (id INT PRIMARY KEY, name STRING, department STRING, hire_date DATE)
        2. courses (id INT PRIMARY KEY, title STRING, required_for STRING)
        3. completions (id INT PRIMARY KEY, employee_id INT, course_id INT, completion_date DATE)

        Relationships:
        - completions.employee_id is a foreign key to employees.id
        - completions.course_id is a foreign key to courses.id

        Best Practices & Security Instructions:
        - Only generate SQL queries that use SELECT statements. Do NOT generate or suggest any queries that modify data (no INSERT, UPDATE, DELETE, MERGE, DROP, ALTER, TRUNCATE, or CREATE).
        - Never expose or request sensitive information such as passwords, emails, or personal identifiers beyond what is present in the schema.
        - Do not attempt to change the database schema or structure.
        - Always validate user intent and clarify ambiguous requests before generating SQL.
        - If a user asks for something outside the scope of the schema or for restricted operations, politely refuse and explain the limitation.
        - Format SQL queries clearly and concisely.
        - When summarizing results, do not fabricate data—only use what is returned from the database.

        Use this schema and these rules to generate SQL queries and answer user questions about employee training, course completions, and departmental analytics.

        Generate only the SQL SELECT query needed to answer the user's last question. Do not include explanations or results. Do not include any format to indicate is a code block.
        """
    },
    {
        "role": "user",
        "content": "List the names of employees who have completed the 'Data Security' course in the last 6 months."
    }
]
deployment_model = "gpt-4o"
response = client.chat.completions.create(
    model=deployment_model,
    messages=messages,
)
print(response.choices[0].message.content)



SELECT e.name 
FROM employees e
JOIN completions c ON e.id = c.employee_id
JOIN courses co ON c.course_id = co.id
WHERE co.title = 'Data Security' 
  AND c.completion_date >= DATEADD(month, -6, CURRENT_DATE());


# Langchain chain

In [10]:

import os
from langchain_openai import AzureChatOpenAI
from dotenv import load_dotenv

load_dotenv()

deployment_model = "gpt-4o"
epam_dial = AzureChatOpenAI(
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    api_version="2025-04-01-preview",
    azure_endpoint="https://ai-proxy.lab.epam.com",
    azure_deployment=deployment_model,
)

result = epam_dial.invoke("A que se refiere elpoema cuando dice que el rio murmura sereno?")

print(result.content)


/Users/ivan_sandin/envs/gen_ai_course/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:26: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


La frase "el río murmura sereno" en un poema generalmente se refiere a la calma y tranquilidad con la que fluye el río. El murmullo del río podría simbolizar la paz, la armonía o un estado de calma en su movimiento. Es posible que el autor esté utilizando esta imagen para evocar una sensación de serenidad en el lector o para reflejar un estado emocional tranquilo. También puede tener un significado simbólico más profundo, dependiendo del contexto del poema. Por ejemplo, puede aludir a una conexión con la naturaleza o a la simple belleza de observar un río fluir plácidamente.


# Chain with openAI Agents

In [13]:

import os
from agents import Agent, Runner, set_tracing_disabled
from agents.extensions.models.litellm_model import LitellmModel
from dotenv import load_dotenv
load_dotenv()

set_tracing_disabled(disabled=True)

os.environ["AZURE_API_KEY"] = os.environ["AZURE_OPENAI_API_KEY"]
os.environ["AZURE_API_BASE"] = "https://ai-proxy.lab.epam.com"
os.environ["AZURE_API_VERSION"] = "2025-04-01-preview"

deployment_model = "azure/gpt-4o"
agent = Agent(
    model=LitellmModel(deployment_model),
    name="Assistant",
    instructions="You are a helpful assistant"
)
result = Runner.run_sync(agent, "Write a haiku about recursion in programming.")
print(result.final_output)


RuntimeError: AgentRunner.run_sync() cannot be called when an event loop is already running.

# Function Calling

In [2]:
import json
def get_weather(location: str, unit: str = "celsius") -> str:
    """Mock function to get weather information""" 
    weather_data = {
        "location": location,
        "temperature": 22 if unit == "celsius" else 72,
        "unit": unit,
        "condition": "sunny",
        "humidity": 60
    }
    return json.dumps(weather_data)

In [3]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get current weather information for a specific location",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "The city and state, e.g. San Francisco, CA"
                    },
                    "unit": {
                        "type": "string",
                        "enum": ["celsius", "fahrenheit"],
                        "description": "The temperature unit to use"
                    }
                },
                "required": ["location"]
            }
        }
    }
]

In [4]:
messages = [
    {
        "role": "system",
        "content": "You are a helpful assistant with access to weather information."
    },
    {
        "role": "user",
        "content": "What is the weather like in Tokyo, Japan?"
    }
]

In [6]:
import os 
from openai import AzureOpenAI
from dotenv import load_dotenv
load_dotenv()

client = AzureOpenAI(
    api_key        = os.environ["AZURE_OPENAI_API_KEY"],
    api_version    = "2025-04-01-preview", 
    azure_endpoint = "https://ai-proxy.lab.epam.com"
)

deployment_model = "gpt-4o"
response = client.chat.completions.create(
    model       = deployment_model,
    messages    = messages,
    tools       = tools,
    tool_choice = "auto"
)

print(response.choices[0].finish_reason)
print(response.choices[0].message.content)
print(response.choices[0].message.tool_calls)

tool_calls
None
[ChatCompletionMessageFunctionToolCall(id='call_7cHPAYLtZB0aGmNGiyoXVH36', function=Function(arguments='{"location":"Tokyo, Japan"}', name='get_weather'), type='function')]


In [7]:
tool = response.choices[0].message.tool_calls[0]
print(tool.id)
print(tool.function.name)
print(tool.function.arguments)

call_7cHPAYLtZB0aGmNGiyoXVH36
get_weather
{"location":"Tokyo, Japan"}


In [8]:
response_message = response.choices[0].message
if response_message.tool_calls:
    messages.append({
        "role": "assistant",
        "content": response_message.content,
        "tool_calls": response_message.tool_calls
    })

In [9]:
if response_message.tool_calls:
    for tool_call in response_message.tool_calls:
        if tool_call.function.name == "get_weather":
            function_args = json.loads(tool_call.function.arguments)
            function_response = get_weather(
                location=function_args.get("location"),
                unit=function_args.get("unit", "celsius")
            )
            messages.append({
                "tool_call_id": tool_call.id,
                "role": "tool",
                "content": function_response
            })

print("Final messages array:")
for msg in messages:
    print(msg)

Final messages array:
{'role': 'system', 'content': 'You are a helpful assistant with access to weather information.'}
{'role': 'user', 'content': 'What is the weather like in Tokyo, Japan?'}
{'role': 'assistant', 'content': None, 'tool_calls': [ChatCompletionMessageFunctionToolCall(id='call_7cHPAYLtZB0aGmNGiyoXVH36', function=Function(arguments='{"location":"Tokyo, Japan"}', name='get_weather'), type='function')]}
{'tool_call_id': 'call_7cHPAYLtZB0aGmNGiyoXVH36', 'role': 'tool', 'content': '{"location": "Tokyo, Japan", "temperature": 22, "unit": "celsius", "condition": "sunny", "humidity": 60}'}


In [10]:
final_response = client.chat.completions.create(
    model    = deployment_model,
    messages = messages,
    tools    = tools,
)
print(final_response.choices[0].message.content)

The current weather in Tokyo, Japan is sunny with a temperature of 22°C and a humidity level of 60%.


# Function call with langchain (tools)

In [11]:
import os
import json
from langchain_openai import AzureChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage
from dotenv import load_dotenv
load_dotenv()

@tool
def get_weather(location: str, unit: str = "celsius") -> str:
    """
    Get current weather information for a specific location

    Args:
        location (str): The city and state, e.g. San Francisco, CA
        unit (str): The temperature unit to use (celsius or fahrenheit)

    """
    return json.dumps({
        "location": location,
        "temperature": 22 if unit == "celsius" else 72,
        "unit": unit,
        "condition": "sunny",
        "humidity": 60
    })

deployment_model = "gpt-4o"
llm = AzureChatOpenAI(
    api_key          = os.environ["AZURE_OPENAI_API_KEY"],
    api_version      = "2025-04-01-preview",
    azure_endpoint   = "https://ai-proxy.lab.epam.com",
    azure_deployment = deployment_model
).bind_tools([get_weather])

msgs = [HumanMessage(content="What's the weather like in New York, NY?")]
res = llm.invoke(msgs)
print("AI Response:", res.content)

for call in getattr(res, "tool_calls", []):
    out = get_weather.invoke(call["args"])
    msgs += [res, ToolMessage(content=out, tool_call_id=call["id"])]
    res = llm.invoke(msgs)
    print("\nFinal AI Response:", res.content)

/Users/ivan_sandin/envs/gen_ai_training/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


AI Response: 

Final AI Response: The weather in New York, NY is currently sunny with a temperature of 22°C and a humidity level of 60%.


# Function calls with agents

In [6]:
import os
import json
import nest_asyncio

from agents import Agent, Runner, set_tracing_disabled, function_tool
from agents.extensions.models.litellm_model import LitellmModel
from dotenv import load_dotenv
load_dotenv()

set_tracing_disabled(disabled=True)

@function_tool
def get_weather(location: str, unit: str = "celsius") -> str:
    """
    Get current weather information for a specific location

    Args:
        location (str): The city and state, e.g., San Francisco, CA
        unit (str): The temperature unit to use (celsius or fahrenheit)

    """
    weather_data = {
        "location": location,
        "temperature": 22 if unit == "celsius" else 72,
        "unit": unit,
        "condition": "sunny",
        "humidity": 60
    }
    return json.dumps(weather_data)

os.environ["AZURE_API_KEY"]     = os.environ["AZURE_OPENAI_API_KEY"]
os.environ["AZURE_API_BASE"]    = "https://ai-proxy.lab.epam.com"
os.environ["AZURE_API_VERSION"] = "2025-04-01-preview"

deployment_model = "azure/gpt-4o"
agent = Agent(
    model        = LitellmModel(deployment_model),
    name         = "Assistant",
    instructions = "You are a helpful assistant with access to weather information.",
    tools        = [get_weather]
)

nest_asyncio.apply()
result = Runner.run_sync(agent, "What is the weather like in Tokyo, Japan?")
print(result.final_output)

RuntimeError: AgentRunner.run_sync() cannot be called when an event loop is already running.

# Image generation

In [7]:

import os
import requests
from dotenv import load_dotenv
load_dotenv()


def get_model_limits(model: dict):
    """Check if the model has usage limits set (indicating availability)."""
    try:
        limits = requests.get(
            f"https://ai-proxy.lab.epam.com/v1/deployments/{model['id']}/limits",
            headers={"Api-Key": os.environ["AZURE_OPENAI_API_KEY"]},
            timeout=20
        ).json()
        minute_limit = limits.get("minuteTokenStats", {}).get("total", 0)
        day_limit = limits.get("dayTokenStats", {}).get("total", 0)
        if minute_limit > 0 or day_limit > 0:
            return {model['id']: {"limits": {"minute": minute_limit, "day": day_limit}}}
    except:
        pass


models = requests.get(
    "https://ai-proxy.lab.epam.com/openai/models",
    headers={"Api-Key": os.environ["AZURE_OPENAI_API_KEY"]},
    timeout=20
).json()["data"]

print("DIAL supports these models:")
for model in models:
    if 'Image Generation' in model['description_keywords']:
        is_available_to_you = get_model_limits(model)
        print("  ->", model['id'], "(available)" if is_available_to_you else "(not available)")

DIAL supports these models:
  -> dall-e-3 (available)
  -> stability.stable-image-core-v1:1 (not available)
  -> stability.stable-image-ultra-v1:1 (not available)
  -> stability.sd3-5-large-v1:0 (not available)
  -> imagegeneration@005 (available)


In [8]:

"""Example of generating an image using DIAL DALL·E-3 model"""
import os
import json
import requests
from dotenv import load_dotenv
load_dotenv()


api_version="2025-04-01-preview"
deployment_model = "dall-e-3"
headers = {
    "api-key": os.environ["AZURE_OPENAI_API_KEY"],
    "Content-Type": "application/json"
}
payload = {
    "messages": [
        {
            "role": "user",
            "content": "Generate a realistic image where a data analyst is working on a laptop in a futuristic office with holographic screens showing graphs and data visualizations."
        }
    ]
}

response = requests.post(
    f"https://ai-proxy.lab.epam.com/openai/deployments/{deployment_model}/chat/completions?api-version={api_version}",
    headers=headers,
    json=payload
).json()

image_data = response["choices"][0]["message"]["custom_content"]['attachments']
print(json.dumps(image_data, indent=3))

image_url = ""
for item in image_data:
    if item['title'] == 'Revised prompt':
        print("Revised prompt:", item['data'])
    elif item['title'] == 'Image':
        image_url = item['url']

print("Image URL:", image_url)

[
   {
      "title": "Revised prompt",
      "data": "A realistic image of a diverse data analyst working on a laptop in a sleek, futuristic office environment. The office features holographic screens displaying colorful graphs and data visualizations floating in mid-air. The analyst, a professional woman of South Asian descent wearing business-casual attire, is focused on her laptop. The scene showcases modern furniture, bright ambient lighting, and a cityscape visible through large, floor-to-ceiling glass windows. The holographic screens show a combination of bar graphs, pie charts, and line graphs with glowing, futuristic designs. The overall mood is innovative, professional, and cutting-edge, with a clear emphasis on technology and progress."
   },
   {
      "title": "Image",
      "type": "image/png",
      "url": "files/2yFqEny55v1gu6DDB4C67cYFg34NTiUYrrXwLTNJviDEocK8WPRT8GrtzwMrSL5zA4/appdata/dall-e-3/images/ab7a89aa049201965331ad412b14c7952e2bbb2a03938d3ffe54563fb2a92c4a.png"

In [9]:

url = f"https://ai-proxy.lab.epam.com/v1/{image_url}"

# Download the file:
response = requests.get(url, headers={"Api-Key": os.environ["AZURE_OPENAI_API_KEY"]})
response.raise_for_status()

# Remove image from the DIAL server after download
delete_response = requests.delete(url, headers={"Api-Key": os.environ["AZURE_OPENAI_API_KEY"]})
delete_response.raise_for_status()

# You can save it to the file:
with open("generated_image.png", "wb") as f:
    f.write(response.content)

## Advanced options image generation

In [10]:

import os
import json
import requests
from dotenv import load_dotenv
load_dotenv()

deployment_model = "dall-e-3"
headers = {
    "Api-Key": os.environ["AZURE_OPENAI_API_KEY"],
}
response = requests.get(
    f"https://ai-proxy.lab.epam.com/v1/deployments/{deployment_model}/configuration",
    headers=headers
).json()

print(json.dumps(response, indent=3))

{
   "title": "Dalle3Config",
   "type": "object",
   "properties": {
      "quality": {
         "title": "Quality",
         "description": "The quality of the image that will be generated.",
         "anyOf": [
            {
               "enum": [
                  "standard",
                  "hd"
               ],
               "type": "string"
            },
            {
               "type": "string"
            }
         ]
      },
      "size": {
         "title": "Size",
         "description": "The size of the generated images.",
         "anyOf": [
            {
               "enum": [
                  "1024x1024",
                  "1792x1024",
                  "1024x1792"
               ],
               "type": "string"
            },
            {
               "type": "string"
            }
         ]
      },
      "style": {
         "title": "Style",
         "description": "The style of the generated images.",
         "anyOf": [
            {
          

In [11]:

payload = {
    "messages": [
        {
            "role": "user",
            "content": "Generate an image of a cat with a hat on a beach"
        }
    ],
    "custom_fields": {
        "configuration": {
            "size": "1792x1024"
        }
    }
}

# Image Recognition

In [12]:

import os
import requests
from dotenv import load_dotenv
load_dotenv()


def get_model_limits(model: dict):
    """Check if the model has usage limits set (indicating availability)."""
    try:
        limits = requests.get(
            f"https://ai-proxy.lab.epam.com/v1/deployments/{model['id']}/limits",
            headers={"Api-Key": os.environ["AZURE_OPENAI_API_KEY"]},
            timeout=20
        ).json()
        minute_limit = limits.get("minuteTokenStats", {}).get("total", 0)
        day_limit = limits.get("dayTokenStats", {}).get("total", 0)
        if minute_limit > 0 or day_limit > 0:
            return {model['id']: {"limits": {"minute": minute_limit, "day": day_limit}}}
    except:
        pass


models = requests.get(
    "https://ai-proxy.lab.epam.com/openai/models",
    headers={"Api-Key": os.environ["AZURE_OPENAI_API_KEY"]},
    timeout=20
).json()["data"]

print("DIAL supports these models:")
for model in models:
    if 'Image Recognition' in model['description_keywords']:
        is_available_to_you = get_model_limits(model)
        print("  ->", model['id'], "(available)" if is_available_to_you else "(not available)")

DIAL supports these models:
  -> gpt-4 (available)
  -> gpt-4o-2024-05-13 (not available)
  -> gpt-4o-2024-08-06 (not available)
  -> gpt-4o-2024-11-20 (not available)
  -> gpt-4o (available)
  -> gpt-4o-mini-2024-07-18 (available)
  -> gpt-4.1-2025-04-14 (not available)
  -> gpt-4.1-nano-2025-04-14 (available)
  -> gpt-4.1-mini-2025-04-14 (available)
  -> gpt-5-2025-08-07 (not available)
  -> gpt-5-2025-08-07-reasoning (not available)
  -> gpt-5-codex-2025-09-15 (not available)
  -> gpt-5-codex-2025-09-15-reasoning (not available)
  -> gpt-5-chat-2025-08-07 (not available)
  -> gpt-5-mini-2025-08-07 (available)
  -> gpt-5-mini-2025-08-07-reasoning (not available)
  -> gpt-5-nano-2025-08-07 (available)
  -> gpt-5-nano-2025-08-07-reasoning (not available)
  -> gpt-5.1-2025-11-13 (not available)
  -> gpt-5.1-2025-11-13-reasoning (not available)
  -> gpt-5.1-chat-2025-11-13 (not available)
  -> gpt-5.1-chat-2025-11-13-reasoning (not available)
  -> gpt-5.1-codex-2025-11-13 (not available)

In [13]:

"""Example of sending an image to DIAL for recognition using GPT-4o."""
import os
import base64
from openai import AzureOpenAI
from dotenv import load_dotenv
load_dotenv()

# Replace `image.png` with your real image file path:
with open("image.png", "rb") as image_file:
    base64_image = base64.b64encode(image_file.read()).decode('utf-8')

# Initialize Azure OpenAI client
client = AzureOpenAI(
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    api_version="2025-04-01-preview",
    azure_endpoint="https://ai-proxy.lab.epam.com/"
)

# Create the chat completion request
response = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": "What is on image?"
                },
                {
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:image/png;base64,{base64_image}"
                    }
                }
            ]
        }
    ]
)

print(response.choices[0].message.content)

The image depicts a modern office setting with floor-to-ceiling glass windows showing a view of city buildings. A person is sitting at a desk with a sleek, futuristic design. On the desk and in the air above, there are semi-transparent holographic displays showcasing data visualizations, graphs, circular interfaces, and other digital elements. The scene conveys a high-tech or futuristic work environment.


## Calling AI Foundry Models with Python

In [15]:
import os
from openai import AzureOpenAI
from dotenv import load_dotenv
load_dotenv()

endpoint = "https://ivan-sandin-9446-resource.cognitiveservices.azure.com/"
deployment = "gpt-5-mini"

subscription_key = os.getenv("FOUNDRY_API_KEY")
api_version = "2024-12-01-preview"

client = AzureOpenAI(
    api_version=api_version,
    azure_endpoint=endpoint,
    api_key=subscription_key,
)

response = client.chat.completions.create(
    messages=[
        {
            "role": "system",
            "content": "You are a helpful assistant.",
        },
        {
            "role": "user",
            "content": "What is the capital of France?",
        }
    ],
    max_completion_tokens=16384,
    model=deployment
)

print(response.choices[0].message.content)

The capital of France is Paris.


## Responses API
Open AI models support Responses API in AI Foundry. Just append /openai/v1/ to your endpoint URL.

In [16]:
import os
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

deployment = "gpt-5-mini"

client = OpenAI(
    api_key=os.getenv("FOUNDRY_API_KEY"),
    base_url="https://ivan-sandin-9446-resource.cognitiveservices.azure.com/openai/v1/",
)

response = client.responses.create(   
  model=deployment,
  input="What is the capital of France?",
)

print(response.model_dump_json(indent=2))

print("\nOutput:")

for x in response.output:
    if x.type == "message":
        print(x.content)

{
  "id": "resp_0e8738e27d8d42e20069509d27dbdc819598e3ca090b9697c9",
  "created_at": 1766890791.0,
  "error": null,
  "incomplete_details": null,
  "instructions": null,
  "metadata": {},
  "model": "gpt-5-mini",
  "object": "response",
  "output": [
    {
      "id": "rs_0e8738e27d8d42e20069509d28110c8195aaa27452b1373889",
      "summary": [],
      "type": "reasoning",
      "content": null,
      "encrypted_content": null,
      "status": null
    },
    {
      "id": "msg_0e8738e27d8d42e20069509d2864d0819592966d92d3dfb746",
      "content": [
        {
          "annotations": [],
          "text": "The capital of France is Paris.",
          "type": "output_text",
          "logprobs": []
        }
      ],
      "role": "assistant",
      "status": "completed",
      "type": "message"
    }
  ],
  "parallel_tool_calls": true,
  "temperature": 1.0,
  "tool_choice": "auto",
  "tools": [],
  "top_p": 1.0,
  "background": false,
  "conversation": null,
  "max_output_tokens": null,
  

## Testing query execution

In [5]:
import os
import re
import pandas as pd
from databricks import sql
# import streamlit as st

def execute_select_query(query: str):
    """
    Executes a SELECT query on Databricks and returns the results as a pandas DataFrame.
    Only SELECT statements are allowed for security reasons.
    """
    # Security: Only allow SELECT queries
    if not re.match(r"^\s*SELECT\s", query, re.IGNORECASE):
        raise ValueError("Only SELECT queries are allowed.")

    # Connect to Databricks using environment variables
    connection = sql.connect(
        server_hostname=os.environ["DATABRICKS_SERVER_HOSTNAME"],
        http_path=os.environ["DATABRICKS_HTTP_PATH"],
        access_token=os.environ["DATABRICKS_TOKEN"]
    )

    try:
        df = pd.read_sql(query, connection)
        return df
    except Exception as e:
        # st.error(f"An error occurred: {e}")
        print(f"An error occurred: {e}")
        return None
    finally:
        connection.close()

In [8]:
# Example usage:
query = "select * FROM employees LIMIT 5"
df = execute_select_query(query)
df
# if df is not None:
#     df.head()

/var/folders/vh/7hc8fzld0332c2_ppdpyvkx00000gn/T/ipykernel_38023/547444275.py:24: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, connection)


,id,name,department,hire_date
0,1,Alice Johnson,Engineering,2024-01-15
1,2,Bob Smith,Marketing,2023-11-20
2,3,Carol Lee,Engineering,2024-03-10
3,4,David Kim,HR,2023-12-05
4,5,Eva Brown,Sales,2024-02-01


In [4]:
import os
import re
import pandas as pd
from databricks import sql

# Connect to Databricks using environment variables
connection = sql.connect(
    server_hostname=os.environ["DATABRICKS_SERVER_HOSTNAME"],
    http_path=os.environ["DATABRICKS_HTTP_PATH"],
    access_token=os.environ["DATABRICKS_TOKEN"]
)

connection.close()

In [14]:
# import os

# print(os.environ["DATABRICKS_SERVER_HOSTNAME"])
# print(os.environ["DATABRICKS_HTTP_PATH"])
# print(os.environ["DATABRICKS_TOKEN"])